# Tutorial: Multislice Propagation and Aperture ROI Modes

This notebook explains the difference between running the hologram simulation with and without multislice free-space propagation, and how aperture regions of interest (ROIs) can be used in different parts of the pipeline.

The key idea:

- **Jones interaction** applies each material slice locally to the beam field.
- **Multislice propagation** additionally propagates the field through free space between material slices.
- **Aperture ROIs** restrict expensive operations to the object/reference-hole support where the mask exposes the sample.

The notebook is didactic first. It includes a compact mode table and an optional tiny pipeline comparison at the end.


## 1. Imports

In [ ]:
import sys
from pathlib import Path
from dataclasses import replace
import time

import h5py
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    %matplotlib widget
except Exception:
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from fomocid import DATA_ROOT
from scattering_calculator.simulation_pipelines.pipelines import (
    HologramPipeline,
    HologramPipelineConfig,
    HologramPipelineRanges,
)


## 2. What changes when `propagate` changes?

In `HologramPipelineConfig`, the flag is:

```python
propagate=False  # local Jones interaction only
propagate=True   # local Jones interaction plus free-space propagation between slices
```

When `propagate=False`, each layer modifies the local Jones vector according to the material dielectric tensor and layer thickness. No angular-spectrum propagation is applied between layers. This is faster and often adequate when the stack is thin and near-field spreading within the stack is negligible.

When `propagate=True`, the field is propagated through free space after each material slice except the last one. This is closer to a multislice treatment: layer transmission, propagation, layer transmission, propagation, and so on. It is more expensive because each free-space step uses FFT-based propagation.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.set_axis_off()
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)

for i, x in enumerate([1.5, 4.0, 6.5]):
    ax.add_patch(plt.Rectangle((x, 2.6), 0.75, 0.9, facecolor="tab:blue", alpha=0.35, edgecolor="black"))
    ax.text(x + 0.375, 3.05, f"layer {i}", ha="center", va="center")
    ax.add_patch(plt.Rectangle((x, 0.6), 0.75, 0.9, facecolor="tab:blue", alpha=0.35, edgecolor="black"))
    ax.text(x + 0.375, 1.05, f"layer {i}", ha="center", va="center")

ax.annotate("Jones", xy=(1.9, 3.05), xytext=(3.2, 3.05), arrowprops={"arrowstyle": "->"})
ax.annotate("Jones", xy=(4.4, 3.05), xytext=(5.7, 3.05), arrowprops={"arrowstyle": "->"})
ax.annotate("Jones", xy=(6.9, 3.05), xytext=(8.2, 3.05), arrowprops={"arrowstyle": "->"})
ax.text(0.2, 3.05, "propagate=False", va="center", fontweight="bold")

ax.annotate("Jones + free-space FFT", xy=(1.9, 1.05), xytext=(3.2, 1.05), arrowprops={"arrowstyle": "->"})
ax.annotate("Jones + free-space FFT", xy=(4.4, 1.05), xytext=(5.7, 1.05), arrowprops={"arrowstyle": "->"})
ax.annotate("Jones", xy=(6.9, 1.05), xytext=(8.2, 1.05), arrowprops={"arrowstyle": "->"})
ax.text(0.2, 1.05, "propagate=True", va="center", fontweight="bold")

ax.set_title("Local Jones-only propagation versus multislice free-space propagation")


## 3. ROI flags in the pipeline

The pipeline has one master ROI switch and several more specific switches.

| Config flag | Affects | Meaning |
|---|---|---|
| `use_roi` | master switch | If `False`, ROI accelerations are disabled even if the specific flags are `True`. |
| `magnetic_pattern_use_roi` | pattern generation | Generate expensive magnetic patterns only around the object-hole region, then paste into a full field. |
| `dielectric_tensor_use_roi` | dielectric tensor and Jones interaction | Build aperture support regions and compute dense/magnetic/vacuum corrections only where the FTH mask opens holes. |
| `dielectric_tensor_compact` | memory representation | Store constant per-layer diagonal terms plus ROI patches instead of one dense `(Nz, Ny, Nx, 2, 2)` array. |
| `multislice_propagation_roi` | free-space propagation between slices | If `propagate=True`, run FFT propagation only inside aperture ROI boxes and apply a plane-wave phase outside. |
| `multislice_propagation_roi_padding_px` | multislice ROI boxes | Enlarge each aperture ROI before local free-space propagation. |

Important nuance: **ROI-only multislice free-space propagation is an approximation**. Free-space propagation is non-local, so a full-field FFT is the more physically faithful option. ROI-only multislice is faster when most of the field is covered by the mask and only aperture regions matter.


## 4. The main operating modes

Use this table as a practical map:

| Mode | `propagate` | `use_roi` | `dielectric_tensor_use_roi` | `multislice_propagation_roi` | What it means |
|---|---:|---:|---:|---:|---|
| Jones-only, full field | `False` | `False` | ignored | ignored | Local layer transmission everywhere. Slowest Jones path, useful as a reference. |
| Jones-only, aperture ROI | `False` | `True` | `True` | ignored | Local Jones interaction uses aperture support regions. Fast and usually the default for thin stacks. |
| Full multislice, full field | `True` | `False` | ignored | `False` | Full-field free-space FFT between slices. Most physically conservative, most expensive. |
| Full multislice with Jones/tensor ROI only | `True` | `True` | `True` | `False` | Jones/tensor work is ROI optimized, but free-space propagation is still full-field. Good accuracy/speed compromise. |
| ROI multislice + Jones/tensor ROI | `True` | `True` | `True` | `True` | Everything aperture-local where possible. Fastest multislice mode, approximate for free-space propagation. |
| ROI multislice requested but no aperture ROIs | `True` | `False` | ignored | `True` | Falls back to full-field free-space propagation because no aperture support regions exist. |

The phrase “using ROIs for multislice but not for Jones” is mostly not a natural pipeline mode, because multislice ROI boxes come from the aperture support regions produced by the dielectric-tensor ROI path. In practice, if you disable aperture support generation, ROI multislice has nothing to crop around and falls back to full-field propagation.


In [ ]:
modes = {
    "jones_full_field": dict(
        propagate=False,
        use_roi=False,
        magnetic_pattern_use_roi=False,
        dielectric_tensor_use_roi=False,
        dielectric_tensor_compact=False,
        multislice_propagation_roi=False,
    ),
    "jones_aperture_roi": dict(
        propagate=False,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=False,
    ),
    "multislice_full_field": dict(
        propagate=True,
        use_roi=False,
        magnetic_pattern_use_roi=False,
        dielectric_tensor_use_roi=False,
        dielectric_tensor_compact=False,
        multislice_propagation_roi=False,
        propagation_padding_px=64,
        propagation_absorber_width_px=32,
        propagation_absorber_strength=4.0,
    ),
    "multislice_full_fft_with_jones_roi": dict(
        propagate=True,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=False,
        propagation_padding_px=64,
        propagation_absorber_width_px=32,
        propagation_absorber_strength=4.0,
    ),
    "multislice_aperture_roi": dict(
        propagate=True,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=True,
        multislice_propagation_roi_padding_px=64,
        propagation_padding_px=32,
        propagation_absorber_width_px=16,
        propagation_absorber_strength=4.0,
    ),
}

for name, settings in modes.items():
    print(name)
    for key, value in settings.items():
        print(f"  {key}: {value}")


## 5. Padding and absorbers for multislice

When `propagate=True`, FFT propagation can suffer from wraparound artifacts at array boundaries. The pipeline exposes:

- `propagation_padding_px`: pad the field before each free-space FFT and crop back afterward.
- `propagation_padding_mode`: passed to NumPy padding; common choices are `"edge"` and `"reflect"`.
- `propagation_absorber_width_px`: apply a smooth absorber at padded edges.
- `propagation_absorber_strength`: stronger values damp edges more.
- `propagation_absorber_profile`: `"cosine"`, `"smoothstep"`, `"quadratic"`, or `"linear"`.

For ROI multislice, there are two different padding ideas:

- `multislice_propagation_roi_padding_px` grows the aperture ROI crop itself.
- `propagation_padding_px` pads inside each crop during FFT propagation.


## 6. Tiny config for optional comparisons

The next cells define a small pipeline config. Running several modes can still take time, so the actual comparison is disabled by default.


In [ ]:
output_folder = DATA_ROOT / "Data" / "multislice_roi_tutorial"
output_folder.mkdir(parents=True, exist_ok=True)

base_config = HologramPipelineConfig(
    recipe="Au(80)/Cr(5)/SiN(80)/Pt(4)Co(6)/Pt(2)",
    sample_name="multislice_roi_tutorial",
    xray_energy=778.0,
    xray_photon_flux=5e8,
    xray_coherence_length=(25e-6, 25e-6),
    detector_shape=(96, 96),
    detector_pixel_size=20e-6,
    detector_distance=0.03,
    detector_center=(48, 48),
    detector_params={
        "readout_noise_average": 20,
        "noise_rms": 3,
        "detector_threshold": 5e4,
        "counts_per_photon": 100,
        "quantum_efficiency": 0.9,
    },
    measurement_config={
        "number_frames": 1,
        "max_counts_per_image": 5e4,
        "exposure_time": 1.0,
    },
    beamstop_method="circular",
    beamstop_distance=0.010,
    beamstop_config={
        "radius": 180e-6,
        "sigma": 10e-6,
        "wire_width": 40e-6,
        "wire_bend": 30e-6,
        "angle": np.deg2rad(25),
        "antialias": 2,
        "seed": 4,
    },
    save_detected_hologram_without_beamstop=True,
    aperture_method="FTH_circular",
    aperture_types=["OH", "RH", "RH"],
    aperture_radii=[95e-9, 7e-9, 9e-9],
    aperture_centers=[(0.0, 0.0), (-140e-9, -125e-9), (125e-9, -140e-9)],
    aperture_sigmas=[4e-9, 2e-9, 2e-9],
    aperture_angles=[0.0, 0.0, 0.0],
    aperture_ellipticities=[1.0, 1.0, 1.0],
    aperture_roughnesses=[0.0, 0.02, 0.02],
    aperture_roughness_modes=[(0, 0), (3, 10), (3, 10)],
    aperture_seeds=[1, 2, 3],
    aperture_top_radius_factors=[1.4, 2.0, 2.0],
    illumination_function="gaussian",
    illumination_center=(0.0, 0.0),
    illumination_focus_distance=1e-3,
    illumination_fwhm=0.45e-6,
    pattern_type="saturated_pattern",
    pattern_config={"saturation": 1.0},
    oversampling=2,
    random_seed=0,
)


## 7. Optional: run selected modes

Set `RUN_COMPARISON = True` to generate one small HDF5 file per selected mode. For a first run, compare only two or three modes.


In [ ]:
RUN_COMPARISON = False
selected_modes = [
    "jones_aperture_roi",
    "multislice_full_fft_with_jones_roi",
    "multislice_aperture_roi",
]

results = {}
if RUN_COMPARISON:
    for mode_name in selected_modes:
        cfg = replace(base_config, **modes[mode_name])
        output_path = output_folder / f"{mode_name}.h5"
        t0 = time.time()
        HologramPipeline(
            config=cfg,
            ranges=HologramPipelineRanges(),
            output_path=output_path,
            n_samples=1,
            verbose=False,
        ).run()
        elapsed = time.time() - t0
        results[mode_name] = {"path": output_path, "seconds": elapsed}
        print(f"{mode_name}: {elapsed:.2f} s -> {output_path}")
else:
    print("Comparison run is disabled. Set RUN_COMPARISON = True to generate mode outputs.")


## 8. Load and compare outputs

The comparison below displays both helicity **differences** (`CR - CL`) and helicity **sums** (`CR + CL`) for the modes you ran. Differences emphasize magnetic contrast; sums emphasize charge/background structure. The same idea is applied to detected holograms and complex exit waves.


In [ ]:
def first_frame(array):
    array = np.asarray(array)
    return array[0] if array.ndim == 3 else array


def load_mode_output(path):
    path = Path(path)
    if not path.exists():
        return None
    with h5py.File(path, "r") as h5:
        grp = h5["00000"]
        return {
            "CR_detected": first_frame(grp["CR/detected"][()]),
            "CL_detected": first_frame(grp["CL/detected"][()]),
            "CR_exit": first_frame(grp["CR/exit_wave"][()]),
            "CL_exit": first_frame(grp["CL/exit_wave"][()]),
            "beamstop": grp["beamstop_mask"][()] if "beamstop_mask" in grp else None,
            "supportmask": grp["supportmask"][()] if "supportmask" in grp else None,
        }


def robust_limits(image, percentiles=(1, 99)):
    data = np.asarray(image)
    vmin, vmax = np.nanpercentile(data, percentiles)
    if np.isclose(vmin, vmax):
        vmin, vmax = float(np.nanmin(data)), float(np.nanmax(data))
    if np.isclose(vmin, vmax):
        vmin, vmax = vmin - 0.5, vmax + 0.5
    return vmin, vmax


loaded = {
    mode_name: load_mode_output(output_folder / f"{mode_name}.h5")
    for mode_name in selected_modes
}
loaded = {name: data for name, data in loaded.items() if data is not None}

if not loaded:
    print("No comparison outputs found yet. Set RUN_COMPARISON = True in the previous section.")
else:
    fig, axes = plt.subplots(len(loaded), 5, figsize=(17, 3.5 * len(loaded)))
    if len(loaded) == 1:
        axes = axes[np.newaxis, :]
    for row, (mode_name, data) in enumerate(loaded.items()):
        detected_diff = data["CR_detected"] - data["CL_detected"]
        detected_sum = data["CR_detected"] + data["CL_detected"]
        exit_diff = data["CR_exit"] - data["CL_exit"]
        exit_sum = data["CR_exit"] + data["CL_exit"]
        panels = [
            (detected_diff, "detected CR - CL", "RdBu_r"),
            (detected_sum, "detected CR + CL", "magma"),
            (np.abs(exit_diff), "|exit wave CR - CL|", "inferno"),
            (np.abs(exit_sum), "|exit wave CR + CL|", "viridis"),
            (data["supportmask"], "support mask", "gray"),
        ]
        for ax, (image, title, cmap) in zip(axes[row], panels):
            vmin, vmax = robust_limits(image)
            ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(f"{mode_name}\n{title}")
            ax.set_axis_off()


## 9. Recommended choices

For most users:

- Start with `propagate=False`, `use_roi=True`, `dielectric_tensor_use_roi=True`, `dielectric_tensor_compact=True`.
- Turn on `propagate=True` when the sample stack is thick enough that propagation between layers matters.
- With `propagate=True`, first try `multislice_propagation_roi=False` so free-space propagation remains full-field.
- Use `multislice_propagation_roi=True` for large sweeps where speed matters and the aperture holes occupy only a small part of the sample plane.
- Increase `multislice_propagation_roi_padding_px` if ROI-only multislice creates edge artifacts around apertures.
- Use `use_roi=False` for reference/debugging runs, small arrays, or when you want to compare against the most direct full-field calculation.

The safest comparison workflow is to run one configuration in two modes, compare CR-CL holograms and exit waves, and only then launch a large sweep.
